# Train one LoRA adapter on Colab

Runtime → Change runtime type → **T4 GPU**. Put `train_lora.py` and `questions.csv` in `MyDrive/lora_runs/`
(once), fill in the parameters, then Runtime → Run all.

One run = one adapter for one subject. The adapter (~70 MB) is pushed to a private Hugging Face repo;
the base model is downloaded here and discarded with the session. Training energy is logged with CodeCarbon
and saved inside the adapter repo as `training_summary.json`. Checkpoints go to Drive under `lora_runs/<repo>/`.

In [ ]:
#@title Parameters
BASE = "Qwen/Qwen2.5-3B-Instruct"   #@param ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-7B-Instruct"]
SUBJECT = "math"                   #@param ["math", "code", "knowledge"]
HF_USER = "your-hf-username"       #@param {type:"string"}
N_EXAMPLES = 2000                  #@param {type:"integer"}
EPOCHS = 2                         #@param {type:"number"}
RANK = 16                          #@param {type:"integer"}
# FOUR_BIT: keep False for 1.5B/3B so the adapter matches the fp16 base the Mac serves; True for the 7B (QLoRA)
FOUR_BIT = False                   #@param {type:"boolean"}
# COUNTRY: ISO-3 code for the training-energy CO2 factor
COUNTRY = "IND"                    #@param {type:"string"}

assert HF_USER != "your-hf-username", "set HF_USER to your Hugging Face username"
REPO = f"{HF_USER}/{BASE.split('/')[-1].lower()}-lora-{SUBJECT}"
print("adapter will be pushed to:", REPO)

In [ ]:
#@title Install (pinned so the adapter loads on the Mac with the same peft major version)
# Colab preinstalls torchao 0.10; recent peft raises on any torchao older than 0.16 even though we
# never use it, so remove it. The pip 'dependency conflict' lines about diffusers/gradio are harmless.
!pip -q uninstall -y torchao
!pip -q install "transformers>=4.45,<5" "peft>=0.13" "datasets>=2.20" "accelerate" "codecarbon>=2.8" "huggingface_hub" bitsandbytes
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
#@title Hugging Face token (WRITE token; accept the base model's licence first if it is gated)
# The token goes into the HF_TOKEN environment variable, which the library and the training
# subprocess read directly. This avoids the named-token store in /root/.cache, which is wiped
# whenever the runtime is recycled ("Token '<name>' not found in stored_tokens").
import os
from getpass import getpass
tok = None
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")          # Colab secret named HF_TOKEN (key icon, left sidebar)
    print("token taken from the HF_TOKEN secret")
except Exception:
    pass
if not tok:
    tok = getpass("Paste your Hugging Face WRITE token: ").strip()
assert tok.startswith("hf_"), "that does not look like a Hugging Face token"
os.environ["HF_TOKEN"] = tok
from huggingface_hub import whoami
print("logged in as", whoami()["name"])

In [ ]:
#@title Mount Drive (script, evaluation bank and checkpoints live there)
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/lora_runs"   #@param {type:"string"}
OUT = os.path.join(DRIVE_DIR, REPO.split("/")[-1])
os.makedirs(OUT, exist_ok=True)
print("outputs ->", OUT)

In [ ]:
#@title Copy train_lora.py and questions.csv from Drive into the session
# Keep both files in DRIVE_DIR (set above). questions.csv is optional: if present, any training row
# whose question is in the evaluation bank is dropped. Fallback: upload them via the Files panel.
import os, shutil
for f in ("train_lora.py", "questions.csv"):
    src = os.path.join(DRIVE_DIR, f)
    if os.path.exists(src) and not os.path.exists(f):
        shutil.copy(src, f)
assert os.path.exists("train_lora.py"), f"train_lora.py not found in {DRIVE_DIR} or the session"
print("train_lora.py ready;", "questions.csv ready" if os.path.exists("questions.csv") else "questions.csv missing (exclusion filter skipped)")

In [ ]:
#@title Train (3B on T4: ~1-2 h for 2,000 examples x 2 epochs; 7B QLoRA: ~3-4 h)
# The command is a Python list (Colab's $var substitution mangled a ! line) and the child's output is
# streamed line by line into this cell and into <OUT>/train.log on Drive, so an error is never hidden.
import os, subprocess, sys, shlex
cmd = [sys.executable, "-u", "train_lora.py",
       "--base", BASE, "--subject", SUBJECT, "--n", str(N_EXAMPLES), "--epochs", str(EPOCHS),
       "--rank", str(RANK), "--out", OUT, "--push-to", REPO, "--country", COUNTRY]
if FOUR_BIT:
    cmd.append("--four-bit")
if os.path.exists("questions.csv"):      # evaluation bank: matching training rows are dropped
    cmd += ["--exclude", "questions.csv"]
print(" ".join(shlex.quote(c) for c in cmd), flush=True)
log_path = os.path.join(OUT, "train.log")
with open(log_path, "w") as log, subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                   text=True, bufsize=1) as proc:
    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
print(f"
exit status {proc.returncode}; full log at {log_path}")
assert proc.returncode == 0, "training failed -- read the lines above"

In [ ]:
#@title Training summary (also saved inside the adapter repo)
import json
print(json.dumps(json.load(open(f"{OUT}/adapter/training_summary.json")), indent=2))
print("adapter repo: https://huggingface.co/" + REPO)

In [ ]:
#@title Push again (only if training finished but the upload to the Hub failed)
import os
from huggingface_hub import HfApi
adapter_dir = f"{OUT}/adapter"
assert os.path.exists(f"{adapter_dir}/adapter_model.safetensors"), f"no adapter at {adapter_dir}; training did not finish"
api = HfApi()
api.create_repo(REPO, private=True, exist_ok=True)
api.upload_folder(folder_path=adapter_dir, repo_id=REPO)
print("pushed:", "https://huggingface.co/" + REPO)